# Held-out DAS-only adjudication and waveform-review checkpoint

This checkpoint reads the compact adjudication products for all 21 held-out DAS-only candidates. The forced generic and template network scores were evaluated at the exact DAS times, and registered broad regional catalog files were checked without threshold repair, candidate deletion, matching-window sweeps, or family assignment. The targeted DAS windows were then read from the registered HDF5 manifest and summarized with the frozen DAS preprocessing.

This remains a **partial** result: the automated metrics and figure are ready for manual waveform/artifact review. No row is called an earthquake, catalog extension, or repeater family.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "config" / "heldout_das_adjudication.json").is_file())
OUT = ROOT / "outputs" / "heldout_v2" / "adjudication"
partial = pd.read_csv(OUT / "partial_adjudication.csv")
wave = pd.read_csv(OUT / "das_only_waveform_review.csv")
partial_status = json.loads((OUT / "partial_adjudication_status.json").read_text())
wave_status = json.loads((OUT / "das_only_waveform_review_status.json").read_text())
assert len(partial) == len(wave) == 21
assert partial["repeater_family_assignment"].eq("not_assigned").all()
assert wave["repeater_family_assignment"].eq("not_assigned").all()
display(wave.head())


In [ ]:
summary = pd.DataFrame({
    "quantity": [
        "held-out DAS-only rows",
        "generic network crossings at DAS time",
        "template network crossings at DAS time",
        "cached regional catalog associations within 30 s",
        "DAS windows with complete finite coverage",
        "family assignments",
    ],
    "value": [
        len(partial),
        int(partial["generic_network_above_frozen_threshold"].sum()),
        int(partial["template_network_above_frozen_threshold"].sum()),
        int((partial["cached_regional_catalog_status"] != "NO_CACHED_BROAD_REGIONAL_EVENT_WITHIN_30S").sum()),
        int((wave["coverage_status"] == "PASS").sum()),
        0,
    ],
})
display(summary)
print("Forced network/template and cached-catalog checks do not validate any DAS-only event.")
print("This is not a catalog-extension claim; manual waveform/artifact review remains the next gate.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(wave["DAS_coincidence_score_frozen"], wave["recomputed_score_max_pm_1s"], c=wave["strong_block_count_at_candidate"], cmap="viridis", s=45)
axes[0].plot([1.5, 2.5], [1.5, 2.5], "k--", lw=0.8)
axes[0].set(xlabel="frozen candidate score", ylabel="recomputed max score ±1 s", title="DAS score reproducibility")
axes[1].bar(range(len(wave)), wave["strong_block_support_duration_pm_1s"], color="tab:orange")
axes[1].set(xlabel="candidate index", ylabel="duration (s)", title="four-block support duration ±1 s")
fig.tight_layout()
display(fig)


In [ ]:
display(Image(filename=str(OUT / "das_only_waveform_review.png")))
print("Figure is an automated review aid; visual interpretation must be recorded before any event-level decision.")


## Scientific interpretation

All 21 DAS-only rows remain unresolved. Their frozen generic and template scores are below threshold, and none has a cached broad-regional catalog association within 30 seconds. The DAS windows have complete finite coverage and show the expected local block-support metrics, but those facts do not distinguish earthquakes from coherent nonseismic transients. The raw payload is int32 without a declared full-scale limit, so saturation was intentionally not assessed.

The next decision gate is manual review of the saved figure and selected waveforms, followed by interval-stratified adjudication. Family labels, slip rates, and stress-drop estimates remain downstream and are not authorized by this checkpoint.
